# Synthetic Nanosheet Instance Segmentation Demo
# 合成ナノシート インスタンスセグメンテーション デモ

---

## What you will learn / このノートブックで学ぶこと

合成ナノシート画像を使って、画像領域分割の基本的な流れを体験します。

扱うタスクは、画像中の個々のナノシートを分けて検出する **instance segmentation** です。

このデモでは、**3つの難易度レベル**（Easy / Mid / Hard）で、以下の3つの手法を比較します：

| Method | Description |
|--------|-------------|
| **SAM (ViT-H)** | Segment Anything Model — 学習不要のゼロショット基盤モデル |
| **YOLO-seg (pretrained)** | COCO学習済み — ナノシートの学習なし |
| **YOLO-seg (trained)** | 合成データ100枚で fine-tuning — タスク特化モデル |

### Workflow / 学習の流れ

1. Setup — リポジトリ取得・ライブラリ準備
2. Load test images — テスト画像を読み込む
3. Load ground-truth masks — 正解マスクを読み込んで可視化する
4. Load predictions — 各手法の予測結果を読み込む
5. Visualize comparison — 予測結果を並べて比較する
6. Compute metrics — 評価指標を計算する
7. Cross-difficulty analysis — 難易度ごとの性能を棒グラフで比較する

---

**Note:** このデモには実験TEM画像は含まれていません。すべて教材用の合成データです。

## Step 1: Setup

GitHubからリポジトリをクローンし、必要なライブラリをインストールします。

In [ ]:
import os

repo_dir = "/content/nanosheet-segmentation-colab-demo"

if not os.path.exists(repo_dir):
    !git clone https://github.com/fanfanfuzzy/nanosheet-segmentation-colab-demo.git
else:
    print("Repository already exists. Skipping clone.")

%cd /content/nanosheet-segmentation-colab-demo
!pip install -r requirements.txt -q

## Step 2: Load test images / テスト画像を読み込む

3つの難易度のテスト画像を読み込みます。
Beer-Lambert物理モデルに基づく合成画像で、難易度が上がるとノイズが増えコントラストが低下します。

| Difficulty | SNR  | Description |
|-----------|------|-------------|
| Easy      | ≈2.7 | High contrast, low noise |
| Mid       | ≈1.7 | Moderate noise |
| Hard      | ≈1.1 | Low contrast, high noise |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cv2

difficulties = ["easy", "mid", "hard"]

# Load test images for each difficulty
test_images = {}  # {difficulty: [image_array, ...]}
for diff in difficulties:
    img_dir = Path(f"demo_assets/{diff}/test_images")
    imgs = []
    for p in sorted(img_dir.glob("*.png")):
        imgs.append(cv2.imread(str(p), cv2.IMREAD_GRAYSCALE))
    test_images[diff] = imgs
    print(f"{diff}: {len(imgs)} images loaded, shape={imgs[0].shape}")

# Show one sample from each difficulty
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, diff in zip(axes, difficulties):
    ax.imshow(test_images[diff][0], cmap="gray")
    ax.set_title(f"{diff.capitalize()} (image 0)", fontsize=14)
    ax.axis("off")
plt.suptitle("Test Images at Different Difficulty Levels", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 3: Load ground-truth masks / 正解マスクを読み込む

各ピクセルにインスタンスID（0=背景、1,2,3,...=各ナノシート）が割り当てられた **label map** を読み込みます。

これを個々のナノシートごとの **binary mask** に変換し、画像に重ねて可視化します。

In [ ]:
def label_map_to_masks(label_map):
    """Convert label map (H,W) -> list of binary masks."""
    ids = np.unique(label_map)
    ids = ids[ids > 0]  # exclude background
    return [(label_map == i).astype(np.uint8) for i in ids]

def overlay_masks(image, masks, alpha=0.4):
    """Overlay colored instance masks on a grayscale image."""
    rgb = np.stack([image] * 3, axis=-1).astype(np.float32) / 255.0
    colors = plt.cm.tab20(np.linspace(0, 1, max(len(masks), 1)))
    for i, mask in enumerate(masks):
        c = colors[i % len(colors)][:3]
        for ch in range(3):
            rgb[:, :, ch] = np.where(mask > 0,
                                     rgb[:, :, ch] * (1 - alpha) + c[ch] * alpha,
                                     rgb[:, :, ch])
    return (rgb * 255).astype(np.uint8)

# Load GT label maps and convert to masks
gt_masks = {}  # {difficulty: [list_of_masks_for_img0, ...]}
for diff in difficulties:
    gt_dir = Path(f"demo_assets/{diff}/ground_truth")
    masks_list = []
    for lm_path in sorted(gt_dir.glob("label_map_*.npy")):
        lm = np.load(str(lm_path))
        masks_list.append(label_map_to_masks(lm))
    gt_masks[diff] = masks_list
    avg_instances = np.mean([len(m) for m in masks_list])
    print(f"{diff}: {len(masks_list)} images, avg {avg_instances:.1f} instances/image")

# Visualize GT overlays
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, diff in zip(axes, difficulties):
    img = test_images[diff][0]
    masks = gt_masks[diff][0]
    ax.imshow(overlay_masks(img, masks))
    ax.set_title(f"{diff.capitalize()} — GT ({len(masks)} instances)", fontsize=13)
    ax.axis("off")
plt.suptitle("Ground-Truth Instance Masks", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 4: Load predictions / 各手法の予測結果を読み込む

GPUサーバーで事前計算された3手法の予測結果を読み込みます。

- **SAM (ViT-H)**: Automatic Mask Generator (AMG) — 画像全体にグリッドポイントを打ち、全マスク候補を自動生成
- **YOLO-seg pretrained**: COCOデータセットで学習済み（ナノシートの学習なし）
- **YOLO-seg trained**: 各難易度の合成データ100枚で fine-tuning 済み

In [ ]:
methods = {
    "SAM (ViT-H)": "predictions_sam",
    "YOLO pretrained": "predictions_yolo_pretrained",
    "YOLO trained": "predictions_yolo_trained",
}

# Load predictions: {difficulty: {method: [masks_for_img0, ...]}}
predictions = {}
for diff in difficulties:
    predictions[diff] = {}
    for method_label, subdir in methods.items():
        pred_dir = Path(f"demo_assets/{diff}/{subdir}")
        masks_list = []
        for idx in range(len(test_images[diff])):
            lm_path = pred_dir / f"label_map_{idx:04d}.npy"
            if lm_path.exists():
                masks_list.append(label_map_to_masks(np.load(str(lm_path))))
            else:
                masks_list.append([])
        predictions[diff][method_label] = masks_list

# Summary
print(f"{'Difficulty':<10} {'Method':<20} {'Avg detected instances'}")
print("-" * 55)
for diff in difficulties:
    for method_label in methods:
        avg = np.mean([len(m) for m in predictions[diff][method_label]])
        print(f"{diff:<10} {method_label:<20} {avg:.1f}")

## Step 5: Visualize comparison / 予測結果を並べて比較する

各難易度から代表的な画像を選び、5列の比較グリッドを表示します：

`Original | Ground Truth | SAM | YOLO pretrained | YOLO trained`

各カラムのタイトルにインスタンス検出数を表示します。

**注目ポイント：**
- SAMは多くのマスクを生成するが、過分割（over-segmentation）の傾向がある
- YOLO pretrainedはナノシートを学習していないため、ほぼ検出できない
- YOLO trainedは学習データと同じタスクなので最も安定

In [ ]:
sample_indices = [0, 3, 7]  # 3 representative images per difficulty

for diff in difficulties:
    n_samples = min(len(sample_indices), len(test_images[diff]))
    fig, axes = plt.subplots(n_samples, 5, figsize=(25, 5 * n_samples))
    if n_samples == 1:
        axes = axes.reshape(1, -1)

    for row, img_idx in enumerate(sample_indices[:n_samples]):
        img = test_images[diff][img_idx]
        gt = gt_masks[diff][img_idx]
        sam_pred = predictions[diff]["SAM (ViT-H)"][img_idx]
        yolo_pre = predictions[diff]["YOLO pretrained"][img_idx]
        yolo_tr = predictions[diff]["YOLO trained"][img_idx]

        # Column 1: Original
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_title(f"Original (image {img_idx:04d})", fontsize=11)

        # Column 2: Ground Truth
        axes[row, 1].imshow(overlay_masks(img, gt))
        axes[row, 1].set_title(f"Ground Truth ({len(gt)})", fontsize=11)

        # Column 3: SAM
        axes[row, 2].imshow(overlay_masks(img, sam_pred))
        axes[row, 2].set_title(f"SAM ({len(sam_pred)})", fontsize=11)

        # Column 4: YOLO pretrained
        axes[row, 3].imshow(overlay_masks(img, yolo_pre))
        axes[row, 3].set_title(f"YOLO pretrained ({len(yolo_pre)})", fontsize=11)

        # Column 5: YOLO trained
        axes[row, 4].imshow(overlay_masks(img, yolo_tr))
        axes[row, 4].set_title(f"YOLO trained ({len(yolo_tr)})", fontsize=11)

        for c in range(5):
            axes[row, c].axis("off")

    plt.suptitle(f"Difficulty: {diff.upper()}", fontsize=18, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Step 6: Compute evaluation metrics / 評価指標を計算する

各手法の予測を定量的に評価します。以下の5つの指標を計算します：

| Metric | Meaning / 意味 |
|--------|----------------|
| `recall` | 正解のうちモデルが見つけた割合 (IoU≥0.5) |
| `precision` | 予測のうち正解だった割合 (IoU≥0.5) |
| `f1` | Recall と Precision の調和平均 |
| `mean_best_iou` | 各正解に対する最良予測とのIoU平均 |
| `semantic_iou` | インスタンスを区別しないピクセル単位のIoU |

In [ ]:
def compute_iou(mask_a, mask_b):
    """Compute IoU between two binary masks."""
    inter = np.logical_and(mask_a > 0, mask_b > 0).sum()
    union = np.logical_or(mask_a > 0, mask_b > 0).sum()
    return float(inter / union) if union > 0 else 0.0

def eval_instance_metrics(gt_list, pred_list, iou_thresh=0.5):
    """Compute instance-level recall, precision, F1, mean_best_iou, semantic_iou."""
    # --- Instance Recall ---
    if len(gt_list) == 0:
        recall = 1.0
    elif len(pred_list) == 0:
        recall = 0.0
    else:
        matched = sum(1 for gt in gt_list
                      if max(compute_iou(gt, p) for p in pred_list) >= iou_thresh)
        recall = matched / len(gt_list)

    # --- Instance Precision ---
    if len(pred_list) == 0:
        precision = 1.0
    elif len(gt_list) == 0:
        precision = 0.0
    else:
        matched = sum(1 for p in pred_list
                      if max(compute_iou(gt, p) for gt in gt_list) >= iou_thresh)
        precision = matched / len(pred_list)

    # --- F1 ---
    f1 = (2 * recall * precision / (recall + precision)) if (recall + precision) > 0 else 0.0

    # --- Mean Best IoU ---
    if len(gt_list) == 0:
        mbi = 1.0
    elif len(pred_list) == 0:
        mbi = 0.0
    else:
        mbi = float(np.mean([max(compute_iou(gt, p) for p in pred_list) for gt in gt_list]))

    # --- Semantic IoU ---
    h, w = (gt_list[0] if gt_list else pred_list[0]).shape
    gt_fg = np.zeros((h, w), dtype=bool)
    for m in gt_list:
        gt_fg |= (m > 0)
    pred_fg = np.zeros((h, w), dtype=bool)
    for m in pred_list:
        pred_fg |= (m > 0)
    inter = np.logical_and(gt_fg, pred_fg).sum()
    union = np.logical_or(gt_fg, pred_fg).sum()
    siou = float(inter / union) if union > 0 else 1.0

    return {"recall": recall, "precision": precision, "f1": f1,
            "mean_best_iou": mbi, "semantic_iou": siou}

In [ ]:
import pandas as pd

# Evaluate all methods x all difficulties x all images
all_results = []
for diff in difficulties:
    n_images = len(test_images[diff])
    for method_label in methods:
        per_image_metrics = []
        for idx in range(n_images):
            gt = gt_masks[diff][idx]
            pred = predictions[diff][method_label][idx]
            m = eval_instance_metrics(gt, pred)
            per_image_metrics.append(m)
        # Average across images
        avg = {k: np.mean([d[k] for d in per_image_metrics]) for k in per_image_metrics[0]}
        avg["difficulty"] = diff
        avg["method"] = method_label
        all_results.append(avg)

results_df = pd.DataFrame(all_results)

# Display with formatting
display_df = results_df[["difficulty", "method", "recall", "precision", "f1", "mean_best_iou", "semantic_iou"]].copy()
for col in ["recall", "precision", "f1", "mean_best_iou", "semantic_iou"]:
    display_df[col] = display_df[col].map(lambda x: f"{x:.3f}")

print("=" * 80)
print("  Evaluation Results: 3 Methods x 3 Difficulty Levels")
print("=" * 80)
display(display_df)

### Per-image detail / 画像ごとの詳細

1枚のテスト画像に対する詳細な評価結果を見てみましょう。
各ナノシートがどの程度正しく検出されたかを確認します。

In [ ]:
# Detailed view for one image
detail_diff = "mid"
detail_idx = 0

img = test_images[detail_diff][detail_idx]
gt = gt_masks[detail_diff][detail_idx]

print(f"Image: {detail_diff}/image_{detail_idx:04d}")
print(f"Ground truth: {len(gt)} instances\n")

for method_label in methods:
    pred = predictions[detail_diff][method_label][detail_idx]
    m = eval_instance_metrics(gt, pred)
    print(f"--- {method_label} ---")
    print(f"  Detected instances: {len(pred)}")
    print(f"  Recall:       {m['recall']:.3f}  ({round(m['recall'] * len(gt))}/{len(gt)} GT matched)")
    print(f"  Precision:    {m['precision']:.3f}  ({round(m['precision'] * len(pred))}/{len(pred)} predictions correct)")
    print(f"  F1:           {m['f1']:.3f}")
    print(f"  Mean best IoU:{m['mean_best_iou']:.3f}")
    print(f"  Semantic IoU: {m['semantic_iou']:.3f}")
    print()

## Step 7: Cross-difficulty analysis / 難易度ごとの性能比較

4つの主要指標について、難易度×手法の棒グラフを作成します。

**観察ポイント：**
- 難易度が上がるとSAMのrecallが大幅に低下する（ノイズに弱い）
- YOLO trainedは各難易度に応じた学習データで訓練されているため、比較的安定
- YOLO pretrainedはどの難易度でもナノシートを検出できない

→ **タスク特化の学習データの価値**が際立ちます。

In [ ]:
method_labels = list(methods.keys())
colors = ["#2196F3", "#FF9800", "#4CAF50"]
metrics_to_plot = ["recall", "f1", "mean_best_iou", "semantic_iou"]
metric_titles = ["Instance Recall", "Instance F1", "Mean Best IoU", "Semantic IoU"]

fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(5 * len(metrics_to_plot), 5))
x = np.arange(len(difficulties))
width = 0.25

for ax, metric, title in zip(axes, metrics_to_plot, metric_titles):
    for i, (ml, color) in enumerate(zip(method_labels, colors)):
        vals = []
        for diff in difficulties:
            row = results_df[(results_df["difficulty"] == diff) & (results_df["method"] == ml)]
            vals.append(float(row[metric].iloc[0]))
        bars = ax.bar(x + i * width, vals, width, label=ml, color=color, alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{v:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_xlabel("Difficulty")
    ax.set_ylabel(title)
    ax.set_title(title, fontsize=13)
    ax.set_xticks(x + width)
    ax.set_xticklabels([d.capitalize() for d in difficulties])
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=7)

plt.suptitle("Segmentation Performance by Difficulty Level", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### Semantic IoU heatmap / ヒートマップで俯瞰する

Semantic IoU（ピクセル単位のIoU）を手法×難易度のヒートマップで表示します。

In [ ]:
pivot = results_df.pivot(index="method", columns="difficulty", values="semantic_iou")
pivot = pivot[difficulties]  # ensure column order

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")

ax.set_xticks(range(len(difficulties)))
ax.set_xticklabels([d.capitalize() for d in difficulties], fontsize=12)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=12)

for i in range(len(pivot.index)):
    for j in range(len(difficulties)):
        val = pivot.values[i, j]
        ax.text(j, i, f"{val:.3f}", ha="center", va="center",
                fontsize=14, fontweight="bold",
                color="white" if val < 0.5 else "black")

plt.colorbar(im, ax=ax, label="Semantic IoU")
ax.set_title("Semantic IoU: Method x Difficulty", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 8: YOLO-seg training demo / YOLOの学習を体験する

---

ここでは、実際にYOLO-segモデルを30エポック学習させて、
「モデルの学習とは何か」を体験します（Colab GPUで2-3分、CPUで10-15分）。

### What happens in training / 学習の流れ

1. **データ準備**: 合成ナノシート画像8枚（train）+ 2枚（val）をYOLO形式で用意済み
2. **モデル初期化**: `yolo11n-seg.pt`（COCOで事前学習済みの軽量モデル）をダウンロード
3. **Fine-tuning**: ナノシート画像でモデルのパラメータを更新（30エポック）
4. **推論**: 学習したモデルでval画像を推論し、Step 5の比較と同様に可視化

**Note:** CPUでも実行可能です（10-15分）。GPUランタイムなら2-3分です。

```
Runtime → Change runtime type → GPU (optional but faster)
```

### 8a: Install ultralytics / ライブラリインストール

In [ ]:
!pip install ultralytics -q

### 8b: Check training data / 学習データの確認

YOLO形式では、各画像に対応するテキストファイル（`.txt`）にアノテーションが格納されています。

各行: `class_id x1 y1 x2 y2 ... xN yN` （正規化された多角形座標）

In [ ]:
import os

# Show dataset structure
print("=== YOLO Dataset Structure ===")
for split in ["train", "val"]:
    img_dir = f"demo_assets/yolo_dataset/{split}/images"
    lbl_dir = f"demo_assets/yolo_dataset/{split}/labels"
    n_imgs = len(os.listdir(img_dir))
    n_lbls = len(os.listdir(lbl_dir))
    print(f"  {split}: {n_imgs} images, {n_lbls} label files")

# Show a sample label file
print("\n=== Sample label (first 3 lines) ===")
with open("demo_assets/yolo_dataset/train/labels/image_0000.txt") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        parts = line.strip().split()
        print(f"  class={parts[0]}, polygon has {(len(parts)-1)//2} vertices")

# Show dataset config
print("\n=== Dataset config (yolo_dataset.yaml) ===")
with open("demo_assets/yolo_dataset.yaml") as f:
    print(f.read())

### 8c: Train YOLO-seg / 学習を実行する

30エポック学習します（Colab GPUで2-3分、CPUで10-15分）。
実際の研究では150+エポック学習しますが、Lossの収束を体験するには30エポックで十分です。

学習ログには以下が表示されます：
- `box_loss`: 物体検出のBounding Box損失
- `seg_loss`: セグメンテーションマスクの損失
- `cls_loss`: クラス分類の損失

In [ ]:
!yolo segment train \
    model=yolo11n-seg.pt \
    data=demo_assets/yolo_dataset.yaml \
    epochs=30 \
    imgsz=512 \
    batch=4 \
    workers=0

### 8d: Run inference with trained model / 学習したモデルで推論する

学習したモデルを使って、val画像を推論します。
30エポックの学習でナノシートをどの程度検出できるか確認します。

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Find the best model from training
train_dir = Path("runs/segment/train")
best_model = train_dir / "weights" / "best.pt"
if not best_model.exists():
    # Try numbered directory
    for d in sorted(Path("runs/segment").glob("train*"), reverse=True):
        candidate = d / "weights" / "best.pt"
        if candidate.exists():
            best_model = candidate
            break

print(f"Using model: {best_model}")
model = YOLO(str(best_model))

# Run inference on val images
val_images = sorted(Path("demo_assets/yolo_dataset/val/images").glob("*.png"))
results = model(val_images, imgsz=512, conf=0.25)

print(f"\nInference completed on {len(val_images)} images.")
for r, img_path in zip(results, val_images):
    n_masks = len(r.masks.data) if r.masks is not None else 0
    print(f"  {img_path.name}: {n_masks} instances detected")

### 8e: Visualize results / 推論結果を可視化する

左: 正解マスク（Ground Truth）、右: 30エポック学習モデルの予測結果

30エポックの学習でナノシートの検出ができていることを確認しましょう。
（Step 5-7の100エポック学習済みモデルとの差にも注目してください）

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

fig, axes = plt.subplots(len(val_images), 3, figsize=(18, 6 * len(val_images)))
if len(val_images) == 1:
    axes = axes.reshape(1, -1)

for row, (r, img_path) in enumerate(zip(results, val_images)):
    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    h, w = img.shape
    idx = int(img_path.stem.split("_")[-1])

    # Original
    axes[row, 0].imshow(img, cmap="gray")
    axes[row, 0].set_title(f"Original ({img_path.name})", fontsize=12)

    # Ground Truth
    gt_lm = np.load(f"demo_assets/easy/ground_truth/label_map_{idx:04d}.npy")
    gt_masks_list = label_map_to_masks(gt_lm)
    axes[row, 1].imshow(overlay_masks(img, gt_masks_list))
    axes[row, 1].set_title(f"Ground Truth ({len(gt_masks_list)} instances)", fontsize=12)

    # Prediction from 30-epoch model
    pred_masks_list = []
    if r.masks is not None:
        for m in r.masks.data:
            mask = m.cpu().numpy()
            if mask.shape != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            pred_masks_list.append((mask > 0.5).astype(np.uint8))
    axes[row, 2].imshow(overlay_masks(img, pred_masks_list))
    axes[row, 2].set_title(f"YOLO 30-epoch ({len(pred_masks_list)} instances)", fontsize=12)

    for c in range(3):
        axes[row, c].axis("off")

plt.suptitle("Short Training Result: GT vs 30-epoch YOLO-seg", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# Compute metrics
print("\n=== Evaluation (30-epoch model on val) ===")
for r, img_path in zip(results, val_images):
    idx = int(img_path.stem.split("_")[-1])
    gt_lm = np.load(f"demo_assets/easy/ground_truth/label_map_{idx:04d}.npy")
    gt_m = label_map_to_masks(gt_lm)
    pred_m = []
    if r.masks is not None:
        h, w = gt_lm.shape
        for m in r.masks.data:
            mask = m.cpu().numpy()
            if mask.shape != (h, w):
                mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)
            pred_m.append((mask > 0.5).astype(np.uint8))
    metrics = eval_instance_metrics(gt_m, pred_m)
    print(f"  {img_path.name}: recall={metrics['recall']:.3f}, "
          f"precision={metrics['precision']:.3f}, "
          f"semantic_iou={metrics['semantic_iou']:.3f}")

print("\nCompare with the pre-trained results from Step 6 above!")

## Summary / まとめ

このデモで、以下を学びました：

1. **合成ナノシート画像の生成** — Beer-Lambert物理モデルに基づく合成データ作成
2. **3つの難易度レベル** — Easy (SNR≈2.7) / Mid (SNR≈1.7) / Hard (SNR≈1.1)
3. **SAM (ViT-H) ゼロショット** — 学習なしでAutomatic Mask Generatorを使う手法
4. **YOLO-seg (pretrained)** — 一般的な事前学習モデルはナノシートを認識できない
5. **YOLO-seg (trained)** — 合成データ100枚で学習したモデルの性能
6. **難易度による性能変化** — ノイズが増えるとSAMの性能が大幅に低下し、学習済みモデルの価値が際立つ

---

### Sim2real gap

実際の研究では、合成データで高性能だったモデルも実際の画像に適用すると性能が低下する場合があります。
この差は **sim2real gap** と呼ばれます。
本デモでは合成データのみを使い、同じ評価の流れを体験しました。

---

### AI-assisted coding

このリポジトリの一部はAIコーディングアシスタント（Devin）を使って作成されています。

重要なのはAIにコードを書かせることよりも、
**AIが行った変更をGitHub上で確認し、安全に取り込むこと**です。

Pull Requestの「Files changed」タブで差分を確認する習慣をつけましょう。